# Model Development and Training

This notebook demonstrates the complete model development pipeline:
1. Data loading and preprocessing
2. Feature engineering
3. Model selection and training
4. Hyperparameter tuning
5. Model evaluation
6. Model saving

In [ ]:
# Import required libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load_data import DataLoader
from src.data.preprocess import DataPreprocessor
from src.features.build_features import FeatureEngineer
from src.models.train_model import ModelTrainer
from src.visualization.visualize import DataVisualizer

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

## 1. Load and Preprocess Data

In [ ]:
# Load data
loader = DataLoader(data_path='../data/raw')
df = loader.load_psl_dataset()

print(f"Original dataset shape: {df.shape}")
df.head()

In [ ]:
# Preprocess data
preprocessor = DataPreprocessor()

# Handle missing values
df_clean = preprocessor.handle_missing_values(df, strategy='drop')

# Remove duplicates
df_clean = preprocessor.remove_duplicates(df_clean)

print(f"Cleaned dataset shape: {df_clean.shape}")

## 2. Feature Engineering

In [ ]:
# Initialize feature engineer
engineer = FeatureEngineer()

# Example: Create datetime features if date column exists
# df_features = engineer.create_datetime_features(df_clean, 'date_column')

# For now, use cleaned data
df_features = df_clean.copy()

print(f"Feature engineered dataset shape: {df_features.shape}")

## 3. Prepare Data for Modeling

In [ ]:
# Define target and features
# NOTE: Update 'target_column' with your actual target variable name
target_column = 'target'  # Update this

# Check if target column exists
if target_column in df_features.columns:
    # Separate features and target
    X = df_features.drop(target_column, axis=1)
    y = df_features[target_column]
    
    # Select only numerical columns for now
    X = X.select_dtypes(include=[np.number])
    
    print(f"Features shape: {X.shape}")
    print(f"Target shape: {y.shape}")
    print(f"Target distribution:\n{y.value_counts()}")
else:
    print(f"Warning: Target column '{target_column}' not found in dataset.")
    print(f"Available columns: {df_features.columns.tolist()}")
    print("Please update the target_column variable above.")

## 4. Train Initial Model

In [ ]:
# Initialize model trainer
# NOTE: Only run this cell if target column exists
if target_column in df_features.columns:
    trainer = ModelTrainer(model_type='random_forest', random_state=42)
    
    # Split data
    X_train, X_test, y_train, y_test = trainer.split_data(X, y, test_size=0.2)
    
    # Train model
    trainer.train(X_train, y_train)
    
    print("Model training completed!")

## 5. Evaluate Model

In [ ]:
# Evaluate model
if target_column in df_features.columns:
    metrics = trainer.evaluate(X_test, y_test)
    
    print("Model Performance Metrics:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")

In [ ]:
# Classification report
if target_column in df_features.columns:
    print("\nDetailed Classification Report:")
    print(trainer.get_classification_report(X_test, y_test))

## 6. Feature Importance

In [ ]:
# Plot feature importance
if target_column in df_features.columns:
    feature_importance = trainer.get_feature_importance(top_n=15)
    
    visualizer = DataVisualizer(output_dir='../reports/figures')
    visualizer.plot_feature_importance(feature_importance, top_n=15, 
                                       save_name='feature_importance.png')

## 7. Confusion Matrix

In [ ]:
# Plot confusion matrix
if target_column in df_features.columns:
    y_pred = trainer.predict(X_test)
    visualizer.plot_confusion_matrix(y_test, y_pred, save_name='confusion_matrix.png')

## 8. Hyperparameter Tuning (Optional)

In [ ]:
# Hyperparameter tuning
if target_column in df_features.columns:
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5]
    }
    
    # This may take some time
    # trainer.hyperparameter_tuning(X_train, y_train, param_grid, cv=3)
    
    print("Uncomment the above line to run hyperparameter tuning")

## 9. Save Model

In [ ]:
# Save the trained model
if target_column in df_features.columns:
    trainer.save_model('psl_model.pkl', model_path='../models')
    print("Model saved successfully!")

## 10. Save Processed Data

In [ ]:
# Save processed data
preprocessor.save_processed_data(df_features, 'processed_psl_data.csv', 
                                 output_path='../data/processed')
print("Processed data saved!")

## Summary

This notebook demonstrated:
- Data loading and preprocessing
- Feature engineering
- Model training
- Model evaluation
- Feature importance analysis
- Model saving

The trained model can now be deployed using the Flask application.